# Week 11 Lab: MLE for Limited Dependent Variable Models

## Learning Objectives

By the end of this lab, you will be able to:
1. Estimate the linear probability model as a baseline using your `lm_ols` function
2. Write a log-likelihood function for binary choice (probit/logit) models
3. Use closures to set up the objective for numerical optimization
4. Obtain MLE estimates using `Optim`
5. Compute standard errors from the Fisher information matrix

## Application: Female Labor Force Participation

A typical textbook example for probit and logit models consists of labor force participation estimation for women. Consider the `mroz.csv` data set, which contains information on 753 women. You are interested in the following specification:

$$\text{inlf} = \beta_1 + \beta_2 \, \text{educ} + \beta_3 \, \text{exper} + \beta_4 \, \text{expersq} + \beta_5 \, \text{age} + \beta_6 \, \text{kidslt6} + \beta_7 \, \text{kidsge6} + \beta_8 \, \text{nwifeinc} + u$$

The dependent variable `inlf` is a dummy which equals 1 if the woman is in the labor force, and zero otherwise.

Information about all other variables (and their positions) are provided in the accompanying file `mroz.des`.

## Loading packages for this notebook

We will be needing the following packages:

In [ ]:
using LinearAlgebra, Distributions, Optim

# Load our library from earlier weeks (provides lm_ols, lm_inference)
include("emet_8014_functions.jl");

## Loading the data

The dependent variable `inlf` is in column 1. The regressors are spread across columns; consult `mroz.des` for column numbers.

In [ ]:
using DelimitedFiles
data = readdlm("../datasets/mroz.csv", ',');

# dependent variable: labor force participation (column 1)
Y = Vector{Float64}(data[:, 1])

# regressor matrix: educ(6), exper(19), expersq(22), age(5), kidslt6(3), kidsge6(4), nwifeinc(20)
X = Matrix{Float64}(data[:, [6, 19, 22, 5, 3, 4, 20]])
X = hcat(ones(length(Y)), X)  # add constant to front

n, k = size(X)

## Exercise 1

Estimate all coefficients using the linear probability model. Also obtain standard errors.

Use `lm_ols` and `lm_inference` from `emet_8014_functions.jl` (which you wrote in earlier weeks).

Store the coefficient estimates as `beta_lpm` and the standard errors as `se_lpm`.

Hint: Both functions return multiple values — check your earlier notebooks for their signatures.

In [ ]:
beta_lpm = nothing  # coefficient estimates
se_lpm   = nothing  # standard errors

# YOUR CODE HERE

## Exercise 2

Implement a log-likelihood function for probit and logit estimation. Recall from the lecture:

$$L(\beta) = \sum_{i=1}^{n} y_i \ln G(x_i'\beta) + \sum_{i=1}^{n} (1-y_i) \ln (1-G(x_i'\beta))$$

where $G$ is a placeholder for the probit or logit cdf.

Call your function `log_likelihood`. It should take these arguments:

* matrix `x` storing the observations for all regressors;
* vector `y` storing observations on the binary dependent variable;
* univariate distribution `d` from the `Distributions` package (whose cdf serves as $G$);
* vector `b` for the coefficients.

Once you have written `log_likelihood`, write a **closure** called `neg_log_likelihood_closure` and initialize it with your sample data for both the probit and logit cases.

Hint: Recall from week 10 the closure pattern. The idea is the same, but now the closure captures `x`, `y`, and `d`, returning a function of `b` only.

In [ ]:
"""
    log_likelihood(x, y, d, b)

Compute the log-likelihood for a binary choice (LDV) model.

# Arguments
- `x::Matrix`: N*K regressor matrix.
- `y::Vector`: N*1 binary outcome vector (0 or 1).
- `d::UnivariateDistribution`: distribution whose cdf G is the link function
  (e.g. `Normal()` for probit, `Logistic()` for logit).
- `b::Vector`: K*1 coefficient vector.

# Returns
- `Float64`: the log-likelihood value.
"""
function log_likelihood(x, y, d, b)
    # Hint: loop over observations; for each i compute G(x[i,:]'b) using cdf(d, ...)

    error("Not yet implemented")
end

In [ ]:
"""
    neg_log_likelihood_closure(x, y, d)

Return a closure that computes the negative log-likelihood as a function of `b` only.

# Arguments
- `x::Matrix`: N*K regressor matrix (captured by the closure).
- `y::Vector`: N*1 binary outcome vector (captured by the closure).
- `d::UnivariateDistribution`: distribution whose cdf is the link function.

# Returns
- `Function`: a function `b -> -log_likelihood(x, y, d, b)`.
"""
neg_log_likelihood_closure(x, y, d) = nothing  # replace nothing with a closure

# initialize for probit and logit
nll_probit = nothing  # use Normal()
nll_logit  = nothing  # use Logistic()

# YOUR CODE HERE

## Exercise 3

Obtain the MLE under probit and logit. Call them `beta_probit` and `beta_logit`.

(Reminder: `Optim` only implements function **minimization**.)

Hint: Use `optimize` with an appropriate gradient-based method and a vector of zeros as the starting value. Extract the minimizer from the result.

In [ ]:
beta_probit = nothing  # probit MLE
beta_logit  = nothing  # logit MLE

# YOUR CODE HERE

## Exercise 4

Obtain the standard errors under the probit and logit models.

Remember from the lecture:
$\sqrt{N} \left( \widehat{\beta} - \beta \right) \overset{d}{\to} \mathcal{N} \left( 0, I(\beta)^{-1} \right)$, where the Fisher information is:

$$I(\beta) = E \left( S(Y_i | X_i, \beta) \, S(Y_i | X_i, \beta)' \right) = E \left( \frac{g(X_i'\beta)^2}{G(X_i'\beta) \left( 1-G(X_i'\beta) \right)} \cdot X_i X_i' \right)$$

The obvious analog estimator of the Fisher information is:

$$\hat{I}(\hat{\beta}) = \frac{1}{N} \sum_{i=1}^{N} \frac{g(X_i'\hat{\beta})^2}{G(X_i'\hat{\beta}) \left( 1-G(X_i'\hat{\beta}) \right)} \cdot X_i X_i'$$

which is consistent: $\hat{I}(\hat{\beta}) = I(\beta) + o_p(1)$.

The asymptotic variance of $\hat{\beta}$ is then estimated as $\frac{1}{N} \hat{I}(\hat{\beta})^{-1}$.

Write a function `se_ldv(x, b, d)` that returns the vector of standard errors.

Hint: Use `cdf(d, z)` for $G(z)$ and `pdf(d, z)` for $g(z)$. The structure is similar to the log-likelihood: loop over observations, accumulate a K*K matrix.

In [ ]:
"""
    se_ldv(x, b, d)

Compute MLE standard errors for a binary choice model.

Uses the inverse of the estimated Fisher information:
    Var_hat(beta_hat) = (1/N) * I_hat^{-1}

# Arguments
- `x::Matrix`: N*K regressor matrix.
- `b::Vector`: K*1 MLE coefficient estimates.
- `d::UnivariateDistribution`: distribution whose cdf/pdf serve as G/g.

# Returns
- `Vector{Float64}`: K*1 vector of standard errors.
"""
function se_ldv(x, b, d)
    # Hint: for each i, compute the weight g(z)^2 / (G(z)(1-G(z))) and accumulate x_i x_i'
    # Then invert and take sqrt of diagonal

    error("Not yet implemented")
end

In [ ]:
se_probit = nothing  # probit standard errors
se_logit  = nothing  # logit standard errors

# YOUR CODE HERE

## Results comparison

Once you have all estimates and standard errors, put them side by side to compare the three models.

In [ ]:
varnames = ["constant", "educ", "exper", "expersq", "age", "kidslt6", "kidsge6", "nwifeinc"]

using Printf

try
    @printf "%-12s | %18s | %18s | %17s | \n" "Variable" "LPM" "Probit" "Logit"
    @printf "%s\n" "-"^75
    for i in 1:k
        @printf "%-12s | %8.4f (%7.4f) | %8.4f (%7.4f) | %8.4f (%7.4f)| \n" varnames[i] beta_lpm[i] se_lpm[i] beta_probit[i] se_probit[i] beta_logit[i] se_logit[i]
    end
catch e
    println("Complete Exercises 1-4 first, then run this cell.")
end